# Stage 2 - judge (Kaggle **GPU T4 x2 required**, Internet ON)
Set the accelerator BEFORE running. Pilot = 8 files x 20 = 160 judgments; expect about 5-15 min on GPU (guess). The script aborts if the first judgments take over 60 s each. Exports only verdict files (no responses).

In [ ]:
import os, glob, subprocess, torch
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. (The last attempt ran on CPU and timed out after 12 h.)"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
GEN_ZIP = sorted(glob.glob("/kaggle/input/**/gen_*.zip", recursive=True))[0]; print(GEN_ZIP)
JUDGE_HF = "Qwen/Qwen2.5-7B-Instruct"; JUDGE_ID = "qwen25-7b"      # fp16 split over 2 GPUs; if memory errors: "Qwen/Qwen2.5-3B-Instruct" and JUDGE_ID = "qwen25-3b"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs && unzip -qo {GEN_ZIP} -d runs
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

In [ ]:
for d in sorted(glob.glob("runs/*/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF}", shell=True, check=True)

In [ ]:
# outcome table next to the paper (n is only 20 per method in the pilot: read the counts, ignore the pass/fail lines)
for d in sorted(glob.glob("runs/*/seed*")):
    tag = d.split("/")[1]
    subprocess.run(f"python scripts/12_evaluate.py --runs runs/{tag} --judge-id {JUDGE_ID} --size 2B --out results/summary_{tag}.csv || true", shell=True)

In [ ]:
!find runs -name "*.judged_*.jsonl" -o -name "gate_scores_*.csv" | zip -q judged_{JUDGE_ID}.zip -@
!ls -la judged_*.zip
!rm -rf /tmp/REINS